In [2]:
!code /data/aneesh/envgptpatch/lib/python3.10/site-packages/vllm/model_executor/models/gpt_oss.py

In [1]:
1+1

2

In [1]:
MODEL_PATH = "/data/aneesh/gpt_oss_120b"
MAPPING_PATH = "/data/aneesh/aimo/pruning/mappings/expert_mapping_120b_mxfp4_new_large.json"

SAVE_PATH = "./models/fixed_120"

TARGET_NUM_EXPERTS = 120
OVERWRITE_OUTPUT = True



import os 
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"


import json
import math
import os
import shutil
from pathlib import Path
from collections import defaultdict

import torch
from safetensors.torch import safe_open, save_file
from transformers import AutoTokenizer


# Cell 3: helpers

MODEL_PATH = Path(MODEL_PATH)
MAPPING_PATH = Path(MAPPING_PATH)
SAVE_PATH = Path(SAVE_PATH)

def ensure_output_dir(path: Path, overwrite: bool = False):
    if path.exists():
        existing = list(path.iterdir())
        if existing and not overwrite:
            raise RuntimeError(f"Output dir already exists and is non-empty: {path}")
        if overwrite:
            shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def load_json(path: Path):
    with open(path, "r") as f:
        return json.load(f)

def save_json(obj, path: Path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def list_source_shards(model_path: Path):
    index_path = model_path / "model.safetensors.index.json"
    if index_path.exists():
        index = load_json(index_path)
        shard_names = sorted(set(index["weight_map"].values()))
        return shard_names, index
    shard_names = sorted([p.name for p in model_path.glob("*.safetensors")])
    return shard_names, None

def tensor_nbytes(t):
    return t.numel() * t.element_size()

def get_num_layers_from_mapping(mapping_json):
    layer_keys = [k for k in mapping_json.keys() if k.startswith("layer_")]
    layer_ids = sorted(int(k.split("_")[1]) for k in layer_keys if k.split("_")[1].isdigit())
    return max(layer_ids) + 1

def build_selected_experts(mapping_json, target_num_experts):
    num_layers = get_num_layers_from_mapping(mapping_json)
    selected = {}
    old_to_new = {}

    for layer_idx in range(num_layers):
        layer_key = f"layer_{layer_idx}"
        if layer_key not in mapping_json:
            raise KeyError(f"Missing {layer_key} in mapping JSON")

        ranked = mapping_json[layer_key]
        if len(ranked) < target_num_experts:
            raise ValueError(
                f"{layer_key} has only {len(ranked)} ranked experts, "
                f"cannot prune to {target_num_experts}"
            )

        chosen = ranked[:target_num_experts]
        selected[layer_idx] = chosen
        old_to_new[str(layer_idx)] = {str(old_idx): new_idx for new_idx, old_idx in enumerate(chosen)}

    return selected, old_to_new

def is_prunable_key(key: str):
    suffixes = [
        ".mlp.router.weight",
        ".mlp.router.bias",
        ".mlp.experts.gate_up_proj_bias",
        ".mlp.experts.down_proj_bias",
        ".mlp.experts.gate_up_proj_blocks",
        ".mlp.experts.gate_up_proj_scales",
        ".mlp.experts.down_proj_blocks",
        ".mlp.experts.down_proj_scales",
    ]
    return any(key.endswith(s) for s in suffixes)

def extract_layer_idx(key: str):
    parts = key.split(".")
    # expected: model.layers.{i}. ...
    for i in range(len(parts) - 2):
        if parts[i] == "layers":
            return int(parts[i + 1])
    raise ValueError(f"Could not parse layer idx from key: {key}")

def slice_expert_dim0(tensor: torch.Tensor, selected_experts):
    idx = torch.tensor(selected_experts, dtype=torch.long)
    return tensor.index_select(0, idx)

def maybe_adjust_router_bias(key: str, tensor: torch.Tensor, target_num_experts: int):
    if key.endswith(".mlp.router.bias") and target_num_experts < 4:
        adjustment = math.log(4.0 / float(target_num_experts))
        return tensor + adjustment
    return tensor

def rewrite_config(model_path: Path, save_path: Path, target_num_experts: int):
    config_path = model_path / "config.json"
    config = load_json(config_path)

    config["num_local_experts"] = int(target_num_experts)
    config["num_experts_per_tok"] = int(min(4, target_num_experts))

    save_json(config, save_path / "config.json")
    return config

def copy_non_weight_files(model_path: Path, save_path: Path):
    skip_names = {
        "config.json",
        "model.safetensors.index.json",
    }
    for p in model_path.iterdir():
        if p.name in skip_names:
            continue
        if p.suffix == ".safetensors":
            continue
        dst = save_path / p.name
        if p.is_file():
            shutil.copy2(p, dst)
        elif p.is_dir():
            shutil.copytree(p, dst, dirs_exist_ok=True)


# Cell 4: build pruning plan

mapping_json = load_json(MAPPING_PATH)
selected_by_layer, expert_mapping_used = build_selected_experts(mapping_json, TARGET_NUM_EXPERTS)

print("num_layers in mapping:", len(selected_by_layer))
print("target experts:", TARGET_NUM_EXPERTS)
print("layer_0 selected:", selected_by_layer[0][:10], "...")
print("layer_1 selected:", selected_by_layer[1][:10], "...")


# Cell 5: prune and write new safetensor shards

ensure_output_dir(SAVE_PATH, overwrite=OVERWRITE_OUTPUT)

shard_names, source_index = list_source_shards(MODEL_PATH)

new_weight_map = {}
total_size = 0
written_tensor_count = 0
pruned_tensor_count = 0
copied_tensor_count = 0

for shard_name in shard_names:
    src_shard_path = MODEL_PATH / shard_name
    dst_shard_path = SAVE_PATH / shard_name

    out_tensors = {}

    with safe_open(str(src_shard_path), framework="pt", device="cpu") as f:
        for key in f.keys():
            tensor = f.get_tensor(key)

            if is_prunable_key(key):
                layer_idx = extract_layer_idx(key)
                selected = selected_by_layer[layer_idx]
                new_tensor = slice_expert_dim0(tensor, selected)
                new_tensor = maybe_adjust_router_bias(key, new_tensor, TARGET_NUM_EXPERTS)
                pruned_tensor_count += 1
            else:
                new_tensor = tensor
                copied_tensor_count += 1

            out_tensors[key] = new_tensor.contiguous()
            new_weight_map[key] = shard_name
            total_size += tensor_nbytes(new_tensor)
            written_tensor_count += 1

    save_file(out_tensors, str(dst_shard_path))

print("written_tensor_count:", written_tensor_count)
print("pruned_tensor_count:", pruned_tensor_count)
print("copied_tensor_count:", copied_tensor_count)
print("num_output_shards:", len(shard_names))


# Cell 6: write index json, config, tokenizer, and mapping used

index_payload = {
    "metadata": {
        "total_size": int(total_size)
    },
    "weight_map": new_weight_map
}

save_json(index_payload, SAVE_PATH / "model.safetensors.index.json")

new_config = rewrite_config(MODEL_PATH, SAVE_PATH, TARGET_NUM_EXPERTS)

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), trust_remote_code=True)
tokenizer.save_pretrained(str(SAVE_PATH))

copy_non_weight_files(MODEL_PATH, SAVE_PATH)

save_json(expert_mapping_used, SAVE_PATH / "expert_mapping.json")

summary = {
    "source_model_path": str(MODEL_PATH),
    "mapping_path": str(MAPPING_PATH),
    "save_path": str(SAVE_PATH),
    "target_num_experts": int(TARGET_NUM_EXPERTS),
    "num_layers": len(selected_by_layer),
    "num_output_shards": len(shard_names),
    "written_tensor_count": written_tensor_count,
    "pruned_tensor_count": pruned_tensor_count,
    "copied_tensor_count": copied_tensor_count,
}
save_json(summary, SAVE_PATH / "pruning_summary.json")

print("saved to:", SAVE_PATH)
print("config num_local_experts:", new_config["num_local_experts"])
print("config num_experts_per_tok:", new_config["num_experts_per_tok"])
print("index tensors:", len(new_weight_map))


/data/aneesh/env_common/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


num_layers in mapping: 36
target experts: 120
layer_0 selected: [49, 45, 113, 126, 60, 15, 17, 77, 100, 123] ...
layer_1 selected: [117, 102, 10, 60, 12, 99, 109, 11, 107, 66] ...
written_tensor_count: 687
pruned_tensor_count: 288
copied_tensor_count: 399
num_output_shards: 15
saved to: models/fixed_120
config num_local_experts: 120
config num_experts_per_tok: 4
index tensors: 687


In [7]:
import os

def get_dir_size_gb(path):
    total_bytes = 0
    for root, _, files in os.walk(path):
        for f in files:
            fp = os.path.join(root, f)
            if os.path.isfile(fp):
                total_bytes += os.path.getsize(fp)
    return total_bytes / (1024 ** 3)



d = "/data/aneesh/pruning/models/120b_mxfp4_100_new"
# d = "/data/aneesh/pruning/oss_120b_mxfp4"

path = d

print(f"{get_dir_size_gb(path):.2f} GB")


48.36 GB


# TESTS, EDA, DE-BUGGING

In [ ]:
# Cell 7: quick structural smoke check

new_index = load_json(SAVE_PATH / "model.safetensors.index.json")
print("num tensors in index:", len(new_index["weight_map"]))
print("total_size:", new_index["metadata"]["total_size"])

for probe_key in [
    "model.layers.0.mlp.router.weight",
    "model.layers.0.mlp.router.bias",
    "model.layers.0.mlp.experts.gate_up_proj_bias",
    "model.layers.0.mlp.experts.down_proj_bias",
    "model.layers.0.mlp.experts.gate_up_proj_blocks",
    "model.layers.0.mlp.experts.gate_up_proj_scales",
    "model.layers.0.mlp.experts.down_proj_blocks",
    "model.layers.0.mlp.experts.down_proj_scales",
]:
    shard = new_index["weight_map"].get(probe_key)
    if shard is None:
        print("missing:", probe_key)
        continue
    with safe_open(str(SAVE_PATH / shard), framework="pt", device="cpu") as f:
        t = f.get_tensor(probe_key)
        print(f"{probe_key:55s} -> {tuple(t.shape)} {t.dtype}")


In [ ]:
# Cell 8: optional load smoke test

import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

cfg = AutoConfig.from_pretrained(str(SAVE_PATH), trust_remote_code=True)
print("loaded config num_local_experts:", getattr(cfg, "num_local_experts", None))
print("loaded config num_experts_per_tok:", getattr(cfg, "num_experts_per_tok", None))

tok = AutoTokenizer.from_pretrained(str(SAVE_PATH), trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    str(SAVE_PATH),
    device_map="auto",
    trust_remote_code=True,
)

messages = [{"role": "user", "content": "Say hello in one short sentence."}]
inputs = tok.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tok.eos_token_id,
    )

print(tok.decode(out[0], skip_special_tokens=False))


In [ ]:
# Compare two checkpoints structurally:
# - tensor names
# - shard/index integrity
# - shape diffs
# - dtype diffs
# - focused MoE tensor diffs
# - grouped repeating-pattern summary

import json
import re
from pathlib import Path
from collections import Counter, defaultdict
from safetensors.torch import safe_open

BASE_PATH = "/data/aneesh/pruning/oss_20b_mxfp4"
PRUNED_PATH = "/data/aneesh/pruning/models/20b_mxfp4_test"

def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

def canonicalize_key(key: str) -> str:
    key = re.sub(r"\bmodel\.layers\.\d+\b", "model.layers.*", key)
    key = re.sub(r"\blayers\.\d+\b", "layers.*", key)
    return key

def scan_checkpoint_dir(dir_path: str):
    dir_path = Path(dir_path)
    shard_files = sorted(dir_path.glob("*.safetensors"))
    index_path = dir_path / "model.safetensors.index.json"
    index_json = load_json(index_path) if index_path.exists() else None

    tensor_meta = {}
    duplicate_keys = set()
    shard_to_keys = defaultdict(list)

    pattern_counts = Counter()
    pattern_shapes = defaultdict(Counter)
    pattern_dtypes = defaultdict(Counter)

    for shard in shard_files:
        with safe_open(str(shard), framework="pt", device="cpu") as f:
            for key in f.keys():
                tensor = f.get_tensor(key)
                meta = {
                    "shape": tuple(tensor.shape),
                    "dtype": str(tensor.dtype),
                    "file": shard.name,
                    "pattern": canonicalize_key(key),
                }
                if key in tensor_meta:
                    duplicate_keys.add(key)
                tensor_meta[key] = meta
                shard_to_keys[shard.name].append(key)

                pattern = meta["pattern"]
                pattern_counts[pattern] += 1
                pattern_shapes[pattern][meta["shape"]] += 1
                pattern_dtypes[pattern][meta["dtype"]] += 1

    return {
        "dir": str(dir_path),
        "shard_files": [p.name for p in shard_files],
        "index_json": index_json,
        "tensor_meta": tensor_meta,
        "duplicate_keys": sorted(duplicate_keys),
        "shard_to_keys": dict(shard_to_keys),
        "pattern_counts": pattern_counts,
        "pattern_shapes": pattern_shapes,
        "pattern_dtypes": pattern_dtypes,
    }

def validate_index(report):
    idx = report["index_json"]
    if idx is None:
        return {
            "has_index": False,
            "missing_from_index": [],
            "extra_in_index": [],
            "missing_files_from_disk": [],
            "unreferenced_disk_files": [],
        }

    weight_map = idx.get("weight_map", {})
    indexed_keys = set(weight_map.keys())
    real_keys = set(report["tensor_meta"].keys())
    indexed_files = set(weight_map.values())
    real_files = set(report["shard_files"])

    return {
        "has_index": True,
        "missing_from_index": sorted(real_keys - indexed_keys),
        "extra_in_index": sorted(indexed_keys - real_keys),
        "missing_files_from_disk": sorted(indexed_files - real_files),
        "unreferenced_disk_files": sorted(real_files - indexed_files),
        "index_total_size": idx.get("metadata", {}).get("total_size"),
    }

def compare_reports(a, b):
    a_keys = set(a["tensor_meta"].keys())
    b_keys = set(b["tensor_meta"].keys())

    common = sorted(a_keys & b_keys)
    only_a = sorted(a_keys - b_keys)
    only_b = sorted(b_keys - a_keys)

    shape_mismatch = []
    dtype_mismatch = []
    shard_mismatch = []

    for k in common:
        am = a["tensor_meta"][k]
        bm = b["tensor_meta"][k]

        if am["shape"] != bm["shape"]:
            shape_mismatch.append((k, am["shape"], bm["shape"]))
        if am["dtype"] != bm["dtype"]:
            dtype_mismatch.append((k, am["dtype"], bm["dtype"]))
        if am["file"] != bm["file"]:
            shard_mismatch.append((k, am["file"], bm["file"]))

    return {
        "common": common,
        "only_a": only_a,
        "only_b": only_b,
        "shape_mismatch": shape_mismatch,
        "dtype_mismatch": dtype_mismatch,
        "shard_mismatch": shard_mismatch,
    }

def pretty_counter(counter_obj, max_items=8):
    items = counter_obj.most_common(max_items)
    return ", ".join([f"{k} x{v}" for k, v in items])

def is_moe_key(key):
    targets = [
        ".mlp.router.weight",
        ".mlp.router.bias",
        ".mlp.experts.gate_up_proj_bias",
        ".mlp.experts.down_proj_bias",
        ".mlp.experts.gate_up_proj_blocks",
        ".mlp.experts.gate_up_proj_scales",
        ".mlp.experts.down_proj_blocks",
        ".mlp.experts.down_proj_scales",
        ".mlp.experts.gate_up_proj",
        ".mlp.experts.down_proj",
    ]
    return any(key.endswith(x) for x in targets)

base = scan_checkpoint_dir(BASE_PATH)
pruned = scan_checkpoint_dir(PRUNED_PATH)
cmp = compare_reports(base, pruned)

base_idx = validate_index(base)
pruned_idx = validate_index(pruned)

print("=" * 160)
print("INDEX / DIRECTORY SANITY")
print("=" * 160)

for name, report, idx in [
    ("BASE", base, base_idx),
    ("PRUNED", pruned, pruned_idx),
]:
    print(f"\n[{name}] {report['dir']}")
    print("num_shards:", len(report["shard_files"]))
    print("num_tensors:", len(report["tensor_meta"]))
    print("duplicate_keys:", len(report["duplicate_keys"]))
    print("has_index:", idx["has_index"])
    if idx["has_index"]:
        print("missing_from_index:", len(idx["missing_from_index"]))
        print("extra_in_index:", len(idx["extra_in_index"]))
        print("missing_files_from_disk:", len(idx["missing_files_from_disk"]))
        print("unreferenced_disk_files:", len(idx["unreferenced_disk_files"]))
        print("index_total_size:", idx["index_total_size"])

print("\n" + "=" * 160)
print("GLOBAL CHECKPOINT DIFF")
print("=" * 160)
print("common tensors:", len(cmp["common"]))
print("only in base:", len(cmp["only_a"]))
print("only in pruned:", len(cmp["only_b"]))
print("shape mismatches:", len(cmp["shape_mismatch"]))
print("dtype mismatches:", len(cmp["dtype_mismatch"]))
print("shard placement mismatches:", len(cmp["shard_mismatch"]))

print("\n--- only in base (first 100) ---")
for k in cmp["only_a"][:100]:
    print(k)

print("\n--- only in pruned (first 100) ---")
for k in cmp["only_b"][:100]:
    print(k)

print("\n--- dtype mismatches (first 100) ---")
for k, da, db in cmp["dtype_mismatch"][:100]:
    print(f"{k:90s} | base={da} | pruned={db}")

print("\n--- shape mismatches (first 300) ---")
for k, sa, sb in cmp["shape_mismatch"][:300]:
    print(f"{k:90s} | base={sa} | pruned={sb}")

print("\n" + "=" * 160)
print("MOE-ONLY SHAPE DIFF")
print("=" * 160)

moe_shape_diffs = [(k, sa, sb) for (k, sa, sb) in cmp["shape_mismatch"] if is_moe_key(k)]
non_moe_shape_diffs = [(k, sa, sb) for (k, sa, sb) in cmp["shape_mismatch"] if not is_moe_key(k)]

print("moe shape mismatches:", len(moe_shape_diffs))
print("non-moe shape mismatches:", len(non_moe_shape_diffs))

print("\n--- moe shape mismatches ---")
for k, sa, sb in moe_shape_diffs[:300]:
    print(f"{k:90s} | base={sa} | pruned={sb}")

print("\n--- non-moe shape mismatches ---")
for k, sa, sb in non_moe_shape_diffs[:100]:
    print(f"{k:90s} | base={sa} | pruned={sb}")

print("\n" + "=" * 160)
print("REPEATING PATTERN SUMMARY")
print("=" * 160)

all_patterns = sorted(
    set(base["pattern_counts"].keys()) | set(pruned["pattern_counts"].keys())
)

header = (
    f"{'pattern':80s} | "
    f"{'checkpoint':10s} | "
    f"{'count':>5s} | "
    f"{'shapes':55s} | "
    f"{'dtypes':20s}"
)
print(header)
print("-" * len(header))

for pattern in all_patterns:
    rows_printed = 0
    for ckpt_name, report in [("base", base), ("pruned", pruned)]:
        count = report["pattern_counts"].get(pattern, 0)
        if count == 0:
            continue
        shapes = pretty_counter(report["pattern_shapes"][pattern], max_items=8)
        dtypes = pretty_counter(report["pattern_dtypes"][pattern], max_items=4)
        print(
            f"{pattern if rows_printed == 0 else '':80s} | "
            f"{ckpt_name:10s} | "
            f"{count:5d} | "
            f"{shapes:55.55s} | "
            f"{dtypes:20.20s}"
        )
        rows_printed += 1
    if rows_printed > 0:
        print("-" * len(header))

print("\n" + "=" * 160)
print("TARGETED MXFP4/MoE CHECK")
print("=" * 160)

targets = [
    "model.layers.*.mlp.router.weight",
    "model.layers.*.mlp.router.bias",
    "model.layers.*.mlp.experts.gate_up_proj_bias",
    "model.layers.*.mlp.experts.down_proj_bias",
    "model.layers.*.mlp.experts.gate_up_proj_blocks",
    "model.layers.*.mlp.experts.gate_up_proj_scales",
    "model.layers.*.mlp.experts.down_proj_blocks",
    "model.layers.*.mlp.experts.down_proj_scales",
]

for pattern in targets:
    print(f"\n### {pattern}")
    for ckpt_name, report in [("base", base), ("pruned", pruned)]:
        count = report["pattern_counts"].get(pattern, 0)
        shapes = pretty_counter(report["pattern_shapes"].get(pattern, Counter()), max_items=8)
        dtypes = pretty_counter(report["pattern_dtypes"].get(pattern, Counter()), max_items=4)
        print(f"{ckpt_name:10s} | count={count:3d} | shapes={shapes} | dtypes={dtypes}")

print("\n" + "=" * 160)
print("FIRST-LAYER PROBE")
print("=" * 160)

probe_keys = [
    "model.layers.0.mlp.router.weight",
    "model.layers.0.mlp.router.bias",
    "model.layers.0.mlp.experts.gate_up_proj_bias",
    "model.layers.0.mlp.experts.down_proj_bias",
    "model.layers.0.mlp.experts.gate_up_proj_blocks",
    "model.layers.0.mlp.experts.gate_up_proj_scales",
    "model.layers.0.mlp.experts.down_proj_blocks",
    "model.layers.0.mlp.experts.down_proj_scales",
]

for k in probe_keys:
    if k in base["tensor_meta"] and k in pruned["tensor_meta"]:
        print(
            f"{k:80s} | "
            f"base={base['tensor_meta'][k]['shape']} {base['tensor_meta'][k]['dtype']} | "
            f"pruned={pruned['tensor_meta'][k]['shape']} {pruned['tensor_meta'][k]['dtype']}"
        )
    else:
        print("missing probe key:", k)


In [ ]:
# Value-level mapping verification for pruned MXFP4 checkpoint
# Verifies: pruned_tensor[new_idx] == base_tensor[old_idx]
# for router/bias/blocks/scales using expert_mapping.json

import json
from pathlib import Path
from safetensors.torch import safe_open
import torch

BASE_PATH = "/data/aneesh/pruning/oss_20b_mxfp4"
PRUNED_PATH = "/data/aneesh/pruning/models/20b_mxfp4_test"
MAPPING_PATH = "/data/aneesh/pruning/models/20b_mxfp4_test/expert_mapping.json"

# Keep this small for quick smoke test; set to None to test all layers
LAYERS_TO_CHECK = [0, 1, 2, 12, 23]
# Set to None to test all kept experts in each checked layer
MAX_EXPERTS_PER_LAYER = 22

TARGET_KEYS = [
    "model.layers.{layer}.mlp.router.weight",
    "model.layers.{layer}.mlp.router.bias",
    "model.layers.{layer}.mlp.experts.gate_up_proj_bias",
    "model.layers.{layer}.mlp.experts.down_proj_bias",
    "model.layers.{layer}.mlp.experts.gate_up_proj_blocks",
    "model.layers.{layer}.mlp.experts.gate_up_proj_scales",
    "model.layers.{layer}.mlp.experts.down_proj_blocks",
    "model.layers.{layer}.mlp.experts.down_proj_scales",
]

def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

def load_index_map(dir_path):
    idx = load_json(Path(dir_path) / "model.safetensors.index.json")
    return idx["weight_map"]

def fetch_tensor(dir_path, weight_map, key):
    shard = weight_map[key]
    with safe_open(str(Path(dir_path) / shard), framework="pt", device="cpu") as f:
        return f.get_tensor(key)

def tensor_equal(a, b):
    if a.dtype.is_floating_point:
        return torch.equal(a, b), float((a - b).abs().max().item())
    return torch.equal(a, b), 0.0

base_weight_map = load_index_map(BASE_PATH)
pruned_weight_map = load_index_map(PRUNED_PATH)
mapping = load_json(MAPPING_PATH)

if LAYERS_TO_CHECK is None:
    LAYERS_TO_CHECK = sorted(int(k) for k in mapping.keys())

total_checks = 0
failed_checks = []

print("=" * 180)
print("VALUE-LEVEL CHECK")
print("=" * 180)

for layer in LAYERS_TO_CHECK:
    layer_map = mapping[str(layer)]
    old_new_pairs = sorted([(int(old_idx), int(new_idx)) for old_idx, new_idx in layer_map.items()], key=lambda x: x[1])

    if MAX_EXPERTS_PER_LAYER is not None:
        old_new_pairs = old_new_pairs[:MAX_EXPERTS_PER_LAYER]

    print(f"\nLAYER {layer} | kept experts: {len(old_new_pairs)}")

    for key_template in TARGET_KEYS:
        key = key_template.format(layer=layer)
        base_tensor = fetch_tensor(BASE_PATH, base_weight_map, key)
        pruned_tensor = fetch_tensor(PRUNED_PATH, pruned_weight_map, key)

        print(f"  {key}")

        for old_idx, new_idx in old_new_pairs:
            base_slice = base_tensor[old_idx]
            pruned_slice = pruned_tensor[new_idx]

            ok, max_abs_diff = tensor_equal(base_slice, pruned_slice)
            total_checks += 1

            if not ok:
                failed_checks.append({
                    "layer": layer,
                    "key": key,
                    "old_idx": old_idx,
                    "new_idx": new_idx,
                    "max_abs_diff": max_abs_diff,
                    "base_shape": tuple(base_slice.shape),
                    "pruned_shape": tuple(pruned_slice.shape),
                    "dtype": str(base_slice.dtype),
                })
                print(f"    FAIL old={old_idx:2d} -> new={new_idx:2d} | max_abs_diff={max_abs_diff}")
                break
        else:
            print("    OK")

print("\n" + "=" * 180)
print("SUMMARY")
print("=" * 180)
print("total_checks:", total_checks)
print("failed_checks:", len(failed_checks))

if failed_checks:
    print("\nFirst few failures:")
    for item in failed_checks[:20]:
        print(item)
else:
    print("All checked slices matched exactly.")
